pytest==8.3.4
torch==2.4.0
torchvision==0.19.0

In [34]:
!torchrun --nproc_per_node 4 sequential_print.py

W0216 01:23:20.673000 139832508346816 torch/distributed/run.py:779] 
W0216 01:23:20.673000 139832508346816 torch/distributed/run.py:779] *****************************************
W0216 01:23:20.673000 139832508346816 torch/distributed/run.py:779] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0216 01:23:20.673000 139832508346816 torch/distributed/run.py:779] *****************************************
```
Process 0
Process 1
Process 2
Process 3
Process 3
Process 2
Process 1
Process 0
---
Process 0
Process 1
Process 2
Process 3
Process 3
Process 2
Process 1
Process 0
---
Process 0
Process 1
Process 2
Process 3
Process 3
Process 2
Process 1
Process 0
---
Process 0
Process 1
Process 2
Process 3
Process 3
Process 2
Process 1
Process 0
---
Process 0
Process 1
Process 2
Process 3
Process 3
Process 2
Process 1
Process 0
```


In [35]:
!nvidia-smi

Mon Feb 16 01:24:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.28.03              Driver Version: 560.28.03      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A4000               Off |   00000000:3B:00.0 Off |                  Off |
| 41%   28C    P8             14W /  140W |    2332MiB /  16376MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = "2,3"


In [20]:
!pytest test_syncbn.py

============================= test session starts ==============================
platform linux -- Python 3.10.12, pytest-9.0.2, pluggy-1.6.0
rootdir: /home/korenikil/efficient-dl-systems/week03_data_parallel/homework
collected 16 items                                                             

test_syncbn.py ................                                          [100%]

============================= 16 passed in 53.07s ==============================


In [22]:
!python native_ddp_cifar100.py

Files already downloaded and verified
100%|█████████████████████████████████████████| 352/352 [00:18<00:00, 19.45it/s]
{
    "benchmark_name": "native_syncbn_ddp",
    "num_gpus": 2,
    "batch_size_per_gpu": 64,
    "total_batch_size": 256,
    "avg_batch_time_seconds": 0.052098026533018454,
    "peak_memory_mb_per_gpu": 66.1806640625,
    "final_accuracy_percent": 38.82
}


In [26]:
!python custom_bn_cifar100.py

Files already downloaded and verified
Epoch 1/10 | loss: 2.0455: 100%|██████████████| 352/352 [00:17<00:00, 19.70it/s]
Epoch 1 accuracy: 16.70%
Epoch 2/10 | loss: 1.6855: 100%|██████████████| 352/352 [00:16<00:00, 21.24it/s]
Epoch 2 accuracy: 23.10%
Epoch 3/10 | loss: 1.3403: 100%|██████████████| 352/352 [00:16<00:00, 21.00it/s]
Epoch 3 accuracy: 27.34%
Epoch 4/10 | loss: 1.3523: 100%|██████████████| 352/352 [00:16<00:00, 21.08it/s]
Epoch 4 accuracy: 30.22%
Epoch 5/10 | loss: 1.0137: 100%|██████████████| 352/352 [00:16<00:00, 21.26it/s]
Epoch 5 accuracy: 31.44%
Epoch 6/10 | loss: 0.9545: 100%|██████████████| 352/352 [00:17<00:00, 20.21it/s]
Epoch 6 accuracy: 32.66%
Epoch 7/10 | loss: 1.0863: 100%|██████████████| 352/352 [00:17<00:00, 20.55it/s]
Epoch 7 accuracy: 32.96%
Epoch 8/10 | loss: 0.9056: 100%|██████████████| 352/352 [00:17<00:00, 20.48it/s]
Epoch 8 accuracy: 33.98%
Epoch 9/10 | loss: 0.7190: 100%|██████████████| 352/352 [00:16<00:00, 20.77it/s]
Epoch 9 accuracy: 34.14%
Epoch 10

In [ ]:
import pandas as pd
import json

files = [
    "native_bn_report_w2.json",
    "custom_bn_report_w2.json"
]

rows = []
for file in files:
    with open(file, "r") as f:
        rows.append(json.load(f))

df = pd.DataFrame(rows)

df.head()


,benchmark_name,num_gpus,batch_size_per_gpu,total_batch_size,avg_batch_time_seconds,peak_memory_mb_per_gpu,final_accuracy_percent
0,native_syncbn_ddp,2,64,256,0.052098,66.180664,38.82
1,custom_syncbn,2,64,256,0.088536,63.837891,33.40


In [26]:
!python benchmark_butterfly_vs_ring.py

BUTTERFLY vs RING ALL-REDUCE (Measured Peak Memory)
Workers 2, Vector 2 | butterfly | Time: 0.001856s | Peak Memory:      2.8KB | Accuracy diff: 0.000000e+00
Workers 2, Vector 2 | ring      | Time: 0.000775s | Peak Memory:      2.2KB | Accuracy diff: 0.000000e+00
Workers 4, Vector 4 | butterfly | Time: 0.001944s | Peak Memory:      3.7KB | Accuracy diff: 0.000000e+00
Workers 4, Vector 4 | ring      | Time: 0.001940s | Peak Memory:      2.6KB | Accuracy diff: 0.000000e+00
Workers 8, Vector 8 | butterfly | Time: 0.003741s | Peak Memory:      5.3KB | Accuracy diff: 0.000000e+00
Workers 8, Vector 8 | ring      | Time: 0.004213s | Peak Memory:      3.4KB | Accuracy diff: 0.000000e+00
Workers 16, Vector 16 | butterfly | Time: 0.006759s | Peak Memory:      8.7KB | Accuracy diff: 1.159497e-07
Workers 16, Vector 16 | ring      | Time: 0.008560s | Peak Memory:      5.1KB | Accuracy diff: 0.000000e+00
Workers 32, Vector 32 | butterfly | Time: 0.013462s | Peak Memory:     15.7KB | Accuracy diff: 2

In [27]:
!python benchmark_ring_vs_torch.py

Vector 2000, 4 workers | torch all reduce | Time: 0.002414s | Peak Memory:      2.6KB | Accuracy diff: 0.000000e+00
Vector 2000, 4 workers | my all reduce | Time: 0.002241s | Peak Memory:      2.6KB | Accuracy diff: 0.000000e+00
Vector 10000, 4 workers | torch all reduce | Time: 0.002295s | Peak Memory:      2.6KB | Accuracy diff: 0.000000e+00
Vector 10000, 4 workers | my all reduce | Time: 0.001965s | Peak Memory:      2.6KB | Accuracy diff: 0.000000e+00
Vector 100000, 4 workers | torch all reduce | Time: 0.001571s | Peak Memory:      2.6KB | Accuracy diff: 0.000000e+00
Vector 100000, 4 workers | my all reduce | Time: 0.018809s | Peak Memory:      2.6KB | Accuracy diff: 0.000000e+00
Vector 2000, 8 workers | torch all reduce | Time: 0.001945s | Peak Memory:      3.4KB | Accuracy diff: 0.000000e+00
Vector 2000, 8 workers | my all reduce | Time: 0.004532s | Peak Memory:      3.4KB | Accuracy diff: 0.000000e+00
Vector 10000, 8 workers | torch all reduce | Time: 0.004365s | Peak Memory:   